# Convolutional Neural Network

In [1]:
import torch, torchvision, re, os
import numpy as np
from PIL import Image

DATA_DIR = '../data/images/climbs'

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

images, climb_ids, angles, labels = [], [], [], []
for filename in sorted(os.listdir(DATA_DIR)):
    # Images are titled 'id=[id]_angle=[angle]_grade=[grade].png'
    id, angle, grade = re.findall(r'id=(.+)_angle=(.+)_grade=(.+)\.png', filename)[0]
    climb_ids.append(id)
    angles.append(int(angle))
    labels.append(int(grade) - 10) # Standardize to [0, 22]
    
    img = Image.open(os.path.join(DATA_DIR, filename)).convert('RGB')
    images.append(preprocess(img).numpy())
images =  np.stack(images)
angles = np.stack(angles)
grades = np.array(labels, dtype=np.int64)

# Separate train/val/test splits
images_train = images[:80]; angles_train = angles[:80]; grades_train = grades[:80]
images_val = images[80:]; angles_val = angles[80:]; grades_val = grades[80:]

In [ ]:
# (for now) simple 2 layer Convolution with average pooling and 1 fully connected layer
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3, 64, 3, padding='same')
        self.pool1 = torch.nn.AvgPool2d(2, 2)
        self.conv2 = torch.nn.Conv2d(64, 128, 3, padding='same')
        self.pool2 = torch.nn.AvgPool2d(2, 2)
        self.fc = torch.nn.Linear(400897, 23) # 128 * 54 * 58 input features + 1 angle
        self.flatten = torch.nn.Flatten()
        self.relu = torch.nn.ReLU()
    
    def forward(self, X, angles):
        c1 = self.conv1(X)
        r1 = self.relu(c1)
        p1 = self.pool1(r1)
        
        c2 = self.conv2(p1)
        r2 = self.relu(c2)
        p2 = self.pool2(r2)
        
        f = self.flatten(p2)
        f = torch.cat([f, angles], dim=1)
        y = self.fc(f)
        return y

In [3]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [4]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, BatchSampler

# Hyperparameters
batch_size = 64
epsilon = 1E-7

model = CNN().to(device)

# Process dataset
class ImageDataset(Dataset):
    def __init__(self, images, angles, grades):
        self.images = images
        self.angles = np.array([angles])
        self.grades = grades
        
    def __len__(self):
        return len(self.grades)

    def __getitem__(self, idx):
        return (self.images[idx], self.angles[:, idx], self.grades[idx])

# Use a WeightedRandomSampler
# SELECT COUNT(DISTINCT climbs.climbUuid) FROM climbs INNER JOIN climb_stats ON climb_stats.climbUuid = climbs.climbUuid WHERE ascentCount >= 5 AND climb_stats.product_layout_uuid = '10' AND currentDifficultyId >= 10 GROUP BY currentDifficultyId;
grade_counts = [334, 795, 916, 1324, 1609, 2003, 2932, 2501, 2854, 2420, 2933, 2367, 3640, 2749, 2249, 1318, 1453, 995, 664, 246, 46, 1, 0]
num_samples = sum(grade_counts)
class_weights = [0 if grade_counts[i] == 0 else num_samples / grade_counts[i] for i in range(len(grade_counts))]
train_weights = [class_weights[grades_train[i]] for i in range(len(grades_train))]
val_weights = [class_weights[grades_val[i]] for i in range(len(grades_val))]

train_sampler = BatchSampler(WeightedRandomSampler(torch.DoubleTensor(train_weights), len(grades_train)), batch_size, drop_last=False)
train_dataset = ImageDataset(torch.from_numpy(images_train), torch.from_numpy(angles_train), torch.from_numpy(grades_train))
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler)

val_sampler = BatchSampler(WeightedRandomSampler(torch.DoubleTensor(val_weights), len(grades_val)), batch_size, drop_last=False)
val_dataset = ImageDataset(torch.from_numpy(images_val), torch.from_numpy(angles_val), torch.from_numpy(grades_val))
val_loader = DataLoader(val_dataset, batch_sampler=val_sampler)

# test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(Y_test)), batch_size=batch_size)

# Use CCE loss
criterion = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

In [5]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 9296302


In [6]:
import plotly.graph_objects as go
import time

# Taken from Lesson 19
def train_model(model, optimizer, num_epochs, checkpoint_path):
    """Train, keep the checkpoint with the lowest validation loss, and plot the losses."""
    training_losses = []
    validation_losses = []
    best_validation_loss = float('inf')
    best_validation_epoch = -1
    training_start = time.time()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        num_batches = 0
        for images, angles, labels in train_loader:
            images = images.to(device)
            angles = angles.to(device)
            labels = labels.to(device)
            outputs = model(images, angles)
            optimizer.zero_grad() 
            loss = criterion(outputs, labels)
            loss.backward() 
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        training_losses.append(avg_loss)

        # Compute validation loss. Don't need gradients here.
        model.eval()
        with torch.no_grad():
            val_total = 0
            for images, angles, labels in val_loader:
                val_outputs = model(images.to(device), angles.to(device))
                val_total += criterion(val_outputs, labels.to(device)).item() * len(labels)
            validation_loss = val_total / len(images_val)
            validation_losses.append(validation_loss)

        # Save the model if this is the best validation loss so far
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_validation_epoch = epoch
            torch.save(model.state_dict(), checkpoint_path)
            best_marker = ' *'
        else:
            best_marker = ''

        if epoch % (num_epochs / 10) == 0:
            print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {avg_loss:.5f}, Val Loss: {validation_loss:.5f}{best_marker}')

    total_time = time.time() - training_start
    print(f'Average time per epoch: {total_time / num_epochs:.2f}s')
    print(f'Best validation loss: {best_validation_loss:.5f} (epoch {best_validation_epoch})')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=list(range(len(training_losses))), y=training_losses, mode='lines', name='Training Loss'))
    fig.add_trace(go.Scatter(x=list(range(len(validation_losses))), y=validation_losses, mode='lines', name='Validation Loss'))
    fig.update_layout(title="Training and Validation Loss Over Time", xaxis_title="Epoch", yaxis_title="Loss")
    fig.show()


train_model(model, optimizer, num_epochs=100, checkpoint_path='kilter_cnn.pth')

Epoch [0/100], Train Loss: 3.14784, Val Loss: 3.14560 *
Epoch [10/100], Train Loss: 3.11518, Val Loss: 3.14362
Epoch [20/100], Train Loss: 3.11657, Val Loss: 3.14120
Epoch [30/100], Train Loss: 3.08067, Val Loss: 3.12927
Epoch [40/100], Train Loss: 3.04272, Val Loss: 3.13443
Epoch [50/100], Train Loss: 3.03321, Val Loss: 3.08783
Epoch [60/100], Train Loss: 3.00095, Val Loss: 3.13014
Epoch [70/100], Train Loss: 2.98150, Val Loss: 3.04263
Epoch [80/100], Train Loss: 2.95647, Val Loss: 3.05736
Epoch [90/100], Train Loss: 2.94801, Val Loss: 3.07074
Average time per epoch: 0.37s
Best validation loss: 3.01690 (epoch 91)
